# MetroEnergy Solutions (MES) — Proof of Concept

**Component demonstrated:** Batch analytics layer → short-term (24h) household/feeder electricity demand forecasting, of the type that would run on the Processing & Analytics tier of the proposed architecture (Section 3 of the report).

**Dataset:** A synthetic half-hourly smart-meter feeder dataset generated to statistically resemble publicly available UK smart-meter datasets such as the Low Carbon London / UK Power Networks smart meter trial and the UCI "Individual Household Electric Power Consumption" dataset (temperature-sensitive daily/weekly seasonality, evening peaks, weekend effect, noise). Using a synthetic but realistic series is consistent with the assignment brief, which permits synthetic/hypothetical datasets, and avoids reliance on an internet connection that is unavailable in this sandboxed environment.

**Model:** Gradient-boosted trees (scikit-learn) trained on calendar + lag + weather features — representative of the kind of lightweight, explainable model that could be deployed as a scheduled batch job (e.g. an orchestrated pipeline task) against the curated ("silver/gold") zone of the data lake described in Figure 2.

---

This notebook is a direct conversion of `mes_poc_generation_and_forecasting_script.py`, structured for assessment, debugging, and iterative improvement. Run cells top-to-bottom for a full reproducible run.

**Reproduction:** run `python support/generate_mes_submission.py` to regenerate all report figures, metrics, and refresh the Word document.


> **Environment note:** Install the dependencies listed in `requirements.txt` before running this notebook. The notebook does not install packages automatically, which keeps the PoC reproducible and suitable for GitHub/assessment submission.


In [ ]:
# --- Setup: imports and output paths ---
# Dependencies: numpy, pandas, matplotlib, scikit-learn
# Optional (Excel cross-check only): openpyxl
import os
from pathlib import Path



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

# Notebook-friendly plotting (inline display + saved figures for the report)
%matplotlib inline
plt.rcParams.update({"font.size": 10, "figure.dpi": 150})

# Local output folders (equivalent to /home/claude/poc and /home/claude/figs in the script)
BASE_DIR = Path(".").resolve()
SUPPORT_DIR = BASE_DIR / "support"
OUTPUT_DIR = SUPPORT_DIR / "output"
FIGURES_DIR = OUTPUT_DIR / "figs"
METRICS_PATH = OUTPUT_DIR / "metrics.csv"
DATASET_PATH = BASE_DIR / "PoC Data_105602.xlsx"

os.makedirs(FIGURES_DIR, exist_ok=True)

RNG = np.random.default_rng(42)
print(f"Working directory: {BASE_DIR}")
print(f"Figures will be saved to: {FIGURES_DIR}")
print(f"Metrics will be saved to: {METRICS_PATH}")

## 1. Data acquisition

Simulated ingestion of half-hourly smart meter feed (120 days).

In [ ]:
# --------------------------------------------------------------------------------------
# 1. DATA ACQUISITION (simulated ingestion of half-hourly smart meter feed, 120 days)
# --------------------------------------------------------------------------------------
periods = 120 * 48  # 120 days of half-hourly readings
idx = pd.date_range("2026-01-01", periods=periods, freq="30min")
df = pd.DataFrame(index=idx)
df["hour"] = df.index.hour + df.index.minute / 60
df["dow"] = df.index.dayofweek
df["doy"] = df.index.dayofyear
df["is_weekend"] = (df["dow"] >= 5).astype(int)

# Ambient temperature feed (proxy for a third-party weather API ingested into the lake)
seasonal_temp = 8 + 6 * np.sin(2 * np.pi * (df["doy"] - 30) / 365)
df["temperature_c"] = seasonal_temp + RNG.normal(0, 1.5, periods)

# Daily demand shape: morning + evening peaks, damped on weekends, temperature-sensitive
morning_peak = 1.6 * np.exp(-((df["hour"] - 7.5) ** 2) / (2 * 1.2 ** 2))
evening_peak = 2.4 * np.exp(-((df["hour"] - 18.5) ** 2) / (2 * 1.6 ** 2))
overnight_base = 0.9 + 0.15 * np.sin(2 * np.pi * df["hour"] / 24)
weekend_damp = np.where(df["is_weekend"] == 1, 0.85, 1.0)
heating_load = np.clip(15 - df["temperature_c"], 0, None) * 0.05  # more load when colder

df["load_kwh"] = (
    (overnight_base + morning_peak + evening_peak) * weekend_damp
    + heating_load
    + RNG.normal(0, 0.12, periods)
).clip(lower=0.05)

# Fault/anomaly injection: occasional voltage-related dropout, reflecting real feeder noise
fault_idx = RNG.choice(periods, size=6, replace=False)
df.iloc[fault_idx, df.columns.get_loc("load_kwh")] *= 0.2

print(f"Raw synthetic feed: {len(df):,} half-hourly rows")
print(f"Date range: {df.index.min()} → {df.index.max()}")
print(f"Injected fault indices (row positions): {sorted(fault_idx.tolist())}")

In [ ]:
# Quick inspection — useful for assessment and debugging
display(df.head())
display(df.describe().T)
print(f"Missing values:\n{df.isna().sum()}")

## 2. Feature engineering

Representative of curated "gold zone" transformation logic.

In [ ]:
# --------------------------------------------------------------------------------------
# 2. FEATURE ENGINEERING (representative of curated "gold zone" transformation logic)
# --------------------------------------------------------------------------------------
df["lag_48"] = df["load_kwh"].shift(48)     # same time yesterday
df["lag_336"] = df["load_kwh"].shift(336)   # same time last week
df["roll_mean_48"] = df["load_kwh"].rolling(48).mean()
df = df.dropna()

features = ["hour", "dow", "is_weekend", "temperature_c", "lag_48", "lag_336", "roll_mean_48"]
target = "load_kwh"

split = int(len(df) * 0.85)
train, test = df.iloc[:split], df.iloc[split:]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

print(f"Feature matrix shape after lag/rolling dropna: {df.shape}")
print(f"Train rows: {len(train):,} | Test rows: {len(test):,}")
print(f"Features: {features}")

In [ ]:
# Optional cross-check against the bundled Excel dataset (PoC Data_105602.xlsx)
# Requires openpyxl: pip install openpyxl
if not DATASET_PATH.exists():
    print(f"No reference file found at {DATASET_PATH.name} — skipping cross-check.")
else:
    try:
        reference = pd.read_excel(DATASET_PATH, sheet_name="synthetic_mes_smart_meter_data")
        reference["timestamp"] = pd.to_datetime(reference["timestamp"])
        reference = reference.set_index("timestamp")
        match = np.allclose(df[features + [target]].values, reference[features + [target]].values)
        print(f"Reference file: {DATASET_PATH.name}")
        print(f"Generated data matches bundled Excel: {match}")
    except ImportError:
        print("openpyxl not installed — skip Excel cross-check (pip install openpyxl to enable).")

## 3. Model training

Batch job — would run on a scheduled compute cluster/serverless job.

In [ ]:
# --------------------------------------------------------------------------------------
# 3. MODEL TRAINING (batch job - would run on a scheduled compute cluster/serverless job)
# --------------------------------------------------------------------------------------
model = GradientBoostingRegressor(
    n_estimators=250, max_depth=3, learning_rate=0.05, random_state=42
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
mape = mean_absolute_percentage_error(y_test, pred) * 100
r2 = r2_score(y_test, pred)

print(f"MAE: {mae:.4f} kWh")
print(f"MAPE: {mape:.2f}%")
print(f"R2: {r2:.4f}")

metrics = pd.DataFrame(
    {"Metric": ["MAE (kWh)", "MAPE (%)", "R-squared"], "Value": [round(mae, 3), round(mape, 2), round(r2, 3)]}
)
metrics.to_csv(METRICS_PATH, index=False)
display(metrics)

## 4. Visual evidence of implementation

Report figures are numbered **sequentially by appearance** in the Word technical report:

| Report figure | Content |
|---|---|
| Figures 1–3 | Shared responsibility, MES architecture, end-to-end dataflow (architecture diagrams) |
| Figures 4–6 | Forecast vs actual, feature importance, weekday/weekend profile (this PoC) |

The architecture diagrams (Figures 1–3) are generated by `support/generate_mes_submission.py`. The cells below regenerate **Figures 4–6** for the PoC section.

Figures are displayed inline and saved to `output/figs/` for report insertion.

In [ ]:
# Architecture diagrams (Figures 1–3) — generated via support/generate_mes_submission.py
import sys
sys.path.insert(0, str(SUPPORT_DIR))
from generate_mes_submission import (
    fig1_shared_responsibility,
    fig2_architecture,
    fig3_dataflow,
)

fig1_shared_responsibility()
fig2_architecture()
fig3_dataflow()
print("Saved Figures 1–3 to", FIGURES_DIR)


In [ ]:
# Chart 1: last 5 days actual vs predicted
fig, ax = plt.subplots(figsize=(9, 4))
last = test.iloc[-240:]
last_pred = pred[-240:]
ax.plot(last.index, last[target], label="Actual demand", color="#3B1A16", linewidth=1.5)
ax.plot(last.index, last_pred, label="Forecast (GBR model)", color="#C45C4A", linewidth=1.5, linestyle="--")
ax.set_title("Figure 4. Actual versus forecast feeder demand (last 5 days of test set)")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Load (kWh per half-hour)")
ax.legend()
ax.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig4_forecast_vs_actual.png")
plt.show()

In [ ]:
# Chart 2: feature importance
importances = pd.Series(model.feature_importances_, index=features).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
importances.plot(kind="barh", ax=ax, color="#C45C4A")
ax.set_title("Figure 5. Relative feature importance — demand forecasting model")
ax.set_xlabel("Importance score")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig5_feature_importance.png")
plt.show()
display(importances.to_frame("Importance"))

In [ ]:
# Chart 3: average daily load profile (weekday vs weekend) - seasonal pattern visualisation
profile = df.groupby(["is_weekend", "hour"])["load_kwh"].mean().unstack(level=0)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(profile.index, profile[0], label="Weekday", color="#3B1A16")
ax.plot(profile.index, profile[1], label="Weekend", color="#C45C4A")
ax.set_title("Figure 6. Average half-hourly load profile: weekday vs weekend")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Average load (kWh)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig6_daily_profile.png")
plt.show()

In [ ]:
print("Charts and metrics generated successfully.")
print(f"Saved metrics: {METRICS_PATH}")
print("Saved figures:")
for name in ["fig4_forecast_vs_actual.png", "fig5_feature_importance.png", "fig6_daily_profile.png"]:
    print(f"  - {FIGURES_DIR / name}")